# Буткемп, день 11 — модели маркет-мейкинга: reservation price, inventory и где они ломаются

Модель маркет-мейкинга превращает четыре числа в две цены. Модель — простая часть; четыре числа — сложная.
Сегодня мы строим stationary quote engine, сверяем его с точным решением и запускаем на своих L2 mid-путях
с явной моделью филлов, а каждый результат подаём как условный на выбранные параметры интенсивности.

## Что задано

Задание звучит как «построить stationary-движок котирования, проверить против точного решения и
прогнать на собственных L2 mid-путях с явной моделью филлов, докладывая каждый результат как условный на
допущенные параметры филлов». Пункты такие:

1. **Measure what you can** — $\sigma$ в час и по биржам из L2 mids (point-in-time), вариация по часам;
   quoted spread и depth на touch.
2. **Implement and verify** — Avellaneda–Stoikov (AS) и Guéant–Lehalle–Fernandez-Tapia (GLFT) как чистые
   функции; DP-солвер на малой inventory-сетке; воспроизвести «AS vs exact» и GLFT-таблицы.
3. **Simulate honestly** — филлы независимыми Бернулли по сторонам на РЕАЛЬНЫХ mid-путях; sweep
   $A\in\{2,5,10\}$, $k\in\{1,1.5,3\}$; политики fixed, GLFT, AS; certainty equivalent за 60 с.
4. **Break it** — информированные филлы $(\pi,J)$: сдвиг оптимального спреда; лимит $\pm10$ против его
   отсутствия: P&L на филл против markout на филл.
5. **Funding** — терминальная стоимость $f\cdot q$ на допущенном timestamp: shift против времени до funding.
6. **Report** — что измерено, что допущено, чувствительность, пройденные тесты, почему результат — верхняя
   граница.

Выдан один файл L2: `~/Downloads/l2_executable_quotes_100ms_10000.parquet` — top-of-book на сетке 100 мс за
48 часов по трём биржам (Bybit, Okex, Binance) и двум активам (BTC, ETH). Колонки: `exchange`, `asset`,
`event_time`, `best_bid`, `best_ask`, `spread_bps`, `best_bid_amount`, `best_ask_amount`, `mid`. Есть
уменьшенная версия `_1s_10000.parquet` с той же схемой. Для $\sigma$ по 1-секундной сетке берём второй файл,
для симулятора на реальных путях — первый (100 мс даёт достаточно мелкий шаг, чтобы вероятности филлов за шаг
были малы).

Reservation price — это цена, вокруг которой мейкер симметрично ставит котировки с учётом своей позиции
(inventory $q$); в AS это $r = s - q\gamma\sigma^2(T-t)$, то есть при длинной позиции мейкер сдвигает
котировки вниз, чтобы охотнее продавать. Inventory — текущая позиция в контрактах. Fill intensity —
пуассоновская интенсивность прихода филлов; в модели она экспоненциально падает с расстоянием котировки от
mid: $\lambda(\delta) = A e^{-k\delta}$, где $A$ — общий уровень активности, $k$ — насколько быстро убывает
интенсивность филлов с расстоянием.

## Пункт 1. Measure what you can — что вообще измеримо из L2

Из top-of-book видны только три вещи: волатильность mid, quoted spread и глубина на touch. Это же и есть
разведка данных: сначала смотрим схему и охват, потом считаем эти три величины. Интенсивности филлов $A$ и
$k$ из этого файла измерить невозможно — у нас нет ни tape, ни собственных филлов, поэтому всё, что
дальше про $A$ и $k$, будет допущением, а не оценкой. Это центральная оговорка дня.

In [1]:
import numpy as np, pandas as pd, duckdb, warnings
warnings.filterwarnings("ignore")
F100 = "/Users/b.kotov/Downloads/l2_executable_quotes_100ms_10000.parquet"
F1S  = "/Users/b.kotov/Downloads/l2_executable_quotes_1s_10000.parquet"
con = duckdb.connect()
head = con.execute(f"SELECT * FROM read_parquet('{F100}') LIMIT 3").df()
info = con.execute(f"""SELECT count(*) AS n, min(event_time) AS t0, max(event_time) AS t1,
    count(DISTINCT exchange) AS n_ex, count(DISTINCT asset) AS n_as
    FROM read_parquet('{F100}')""").df()
print("3 строки:")
print(head[['exchange','asset','event_time','best_bid','best_ask','spread_bps','mid']].to_string(index=False))
print()
print("всего строк:", int(info.n[0]), "| интервал:", info.t0[0], "->", info.t1[0],
      "| бирж:", int(info.n_ex[0]), "| активов:", int(info.n_as[0]))
print("часы:", round((info.t1[0]-info.t0[0]).total_seconds()/3600, 1))

3 строки:
exchange asset                       event_time  best_bid  best_ask  spread_bps       mid
   Bybit   BTC 2026-07-01 03:00:00.083350+03:00  58603.40  58603.50    0.017064 58603.450
    Okex   ETH 2026-07-01 03:00:00.089859+03:00   1571.35   1571.40    0.318193  1571.375
   Bybit   ETH 2026-07-01 03:00:00.093596+03:00   1571.20   1571.21    0.063645  1571.205

всего строк: 10244979 | интервал: 2026-07-01 03:00:00.083350+03:00 -> 2026-07-03 02:59:59.896158+03:00 | бирж: 3 | активов: 2
часы: 48.0


Файл содержит 10.2 млн строк за 48 часов — на 100-мс сетке это ~172 тысячи точек на пару
«биржа × актив». Волатильность считаем по 1-секундной сетке (файл `_1s`), чтобы не ловить микроструктурный
шум и артефакты нерегулярной сетки: $\sigma$ в bp на $\sqrt{\text{s}}$ — стандартное отклонение
односекундных логарифмических доходностей. Таблица описательная, но $\sigma$, которая войдёт в политику котирования в симуляторе, берётся
строго по прошлому (walk-forward: каждое окно котируется по realized-волатильности предыдущего) —
иначе получим look-ahead.

In [2]:
# sigma по биржам и активам (1s сетка) + вариация по часам
d1 = con.execute(f"""SELECT exchange, asset, event_time, mid FROM read_parquet('{F1S}')
    WHERE mid IS NOT NULL ORDER BY event_time""").df()

print("sigma (bp/sqrt(s)) и разброс по часам:")
rows = []
for ex in ["Bybit", "Okex", "Binance"]:
    for a in ["BTC", "ETH"]:
        s = d1[(d1.exchange == ex) & (d1.asset == a)].set_index("event_time").mid
        r = np.diff(np.log(s.to_numpy())) * 1e4                  # bp за 1s
        rs = pd.Series(r, index=s.index[1:])
        hh = rs.groupby(rs.index.floor("h")).std()
        rows.append((ex, a, np.nanstd(r), hh.min(), hh.median(), hh.max(), hh.max()/hh.min()))
sig = pd.DataFrame(rows, columns=["exchange","asset","sigma","h_min","h_med","h_max","ratio"])
print(sig.round(3).to_string(index=False))

sigma (bp/sqrt(s)) и разброс по часам:


exchange asset  sigma  h_min  h_med  h_max  ratio
   Bybit   BTC  0.840  0.331  0.673  1.512  4.564
   Bybit   ETH  1.104  0.490  0.903  2.417  4.937
    Okex   BTC  0.812  0.317  0.648  1.481  4.676
    Okex   ETH  1.075  0.471  0.864  2.397  5.093
 Binance   BTC  0.820  0.292  0.654  1.509  5.174
 Binance   ETH  1.067  0.445  0.855  2.353  5.284


In [3]:
# quoted spread и глубина на touch (BTC) — из 100ms файла (там есть спред и размеры)
d100s = con.execute(f"""SELECT exchange, mid, spread_bps, best_bid_amount, best_ask_amount
    FROM read_parquet('{F100}') WHERE asset='BTC' AND mid IS NOT NULL""").df()
print("спред (bp) и глубина на touch (контракты), BTC:")
for ex in ["Bybit", "Okex", "Binance"]:
    s = d100s[d100s.exchange == ex]
    print(f"  {ex:8s}: spread median {s.spread_bps.median():.4f} bp  p95 {s.spread_bps.quantile(0.95):.4f} bp | "
          f"bid depth median {s.best_bid_amount.median():.2f}  ask depth median {s.best_ask_amount.median():.2f}")

mid0 = d100s[d100s.exchange == "Bybit"].mid.median()
TICK = 0.1                                  # шаг цены BTC-перпетуала на Bybit
print()
print(f"медианная цена BTC ~ {mid0:.1f}; тик {TICK} = {TICK/mid0*1e4:.4f} bp "
      f"-> спред 0.0166 bp = {0.0166/(TICK/mid0*1e4):.1f} тик(а)")

спред (bp) и глубина на touch (контракты), BTC:


  Bybit   : spread median 0.0166 bp  p95 0.0171 bp | bid depth median 2.40  ask depth median 2.25
  Okex    : spread median 0.0166 bp  p95 0.0171 bp | bid depth median 4.80  ask depth median 4.68


  Binance : spread median 0.0166 bp  p95 0.0171 bp | bid depth median 4.60  ask depth median 4.69

медианная цена BTC ~ 60201.6; тик 0.1 = 0.0166 bp -> спред 0.0166 bp = 1.0 тик(а)


По BTC $\sigma$ почти одинакова на трёх биржах ($0.812$–$0.840$ bp/$\sqrt{\text{s}}$), по
ETH — заметно выше ($1.067$–$1.104$ bp/$\sqrt{\text{s}}$): крипта волатильнее традиционных инструментов, и
ETH примерно на четверть живее BTC. Разброс $\sigma$ по часам внутри одних суток — 4.6–5.3× ($\sigma$ от
~0.3 до ~1.5 bp/$\sqrt{\text{s}}$ по BTC). Это большой разброс: модель предполагает $\sigma$ постоянной, а
рынок меняет её в разы за сутки, что прямо ведёт к вопросу «где модель ломается». Спред BTC зажат на
одном тике ($0.0166$ bp при цене ~$60200$), глубина на touch — единицы контрактов (median 2.2–4.8).
Сопоставим это с масштабом модели ниже: модельный half-spread из эталонного примера — $0.667$ bp, то есть в
~40 раз шире реального спреда BTC. Реальность много плотнее иллюстративного примера; путать эти масштабы
нельзя. Причина контраста — как раз недоизмеренные $(A,k)$: при допущенном $k=1.5$ оптимальный
half-spread $1/k$ и есть $0.667$ bp, а спред в один тик рентабелен котировщику только при много более
высокой интенсивности филлов на этом расстоянии (реальные $A,k$ сильно отличаются от иллюстративных).
Спред — выход модели, а не её вход; сравнивать модельный пример с реальным стаканом напрямую некорректно.

## Гипотеза дня и её проверка

Формулируем гипотезу: closed-form AS считает inventory skew по горизонту $T-t$, но мейкер на 24/7-крипте не
держит позицию до горизонта — филлы разравнивают inventory за секунды. Значит, AS будет систематически и
сильно завышать skew, а правильная асимптотика — stationary-решение (GLFT), которое не зависит от $T-t$
вообще. Проверяем числом: решаем уравнение Гамильтона–Якоби–Беллмана (HJB) точно методом динамического
программирования (DP) на малой inventory-сетке и сравниваем skew при $q=3$ у AS и у точного решения.

Ждём следующего: у AS skew $= 2q\gamma\sigma^2\tau$ растёт с горизонтом, у точного — не зависит от него. Если
гипотеза верна, при $\tau=60$ с расхождение — порядок величины.

## Пункт 2. Implement and verify — AS, GLFT, DP

AS в closed form (для mid $s$, текущей inventory $q$, горизонта $\tau=T-t$):

$$r = s - q\gamma\sigma^2\tau,\qquad
\Delta = \gamma\sigma^2\tau + \frac{2}{\gamma}\ln\!\left(1+\frac{\gamma}{k}\right),\qquad
\text{ask}=r+\frac{\Delta}{2},\;\text{bid}=r-\frac{\Delta}{2}.$$

$A$ не входит в AS-формулу вовсе: параметры $A$ и $k$ независимы, и AS
«видит» из них только $k$; это красный флаг, который всплывёт в sweep. Пишем AS и GLFT как чистые функции и сверяем
эталонные числа ($\sigma=1$, $\gamma=0.01$, $k=1.5$, $\tau=60$).

In [4]:
# чистые функции моделей
def as_quotes(s, q, gamma, sigma, k, tau):
    """AS closed form: возвращает (bid, ask) как абсолютные цены."""
    r = s - q * gamma * sigma**2 * tau
    delta = gamma * sigma**2 * tau + (2.0/gamma)*np.log(1.0 + gamma/k)
    return r - delta/2.0, r + delta/2.0

def glft_quotes(s, q, gamma, sigma, k, A):
    """GLFT stationary: цены (bid, ask). bid стоит под mid на расстоянии
    base + (2q+1)/2*s0, ask над mid на расстоянии base - (2q-1)/2*s0."""
    base = (1.0/gamma)*np.log(1.0 + gamma/k)
    s0 = np.sqrt(sigma**2 * gamma / (2.0*A*k) * (1.0 + gamma/k)**(1.0 + k/gamma))
    return s - (base + (2*q + 1)/2.0*s0), s + (base - (2*q - 1)/2.0*s0)

GAMMA, SIGMA0, KK, AA, TAU = 0.01, 1.0, 1.5, 5.0, 60.0
print("=== AS эталон (sigma=1, gamma=0.01, k=1.5, tau=60) ===")
print("gamma*sigma^2*tau         =", GAMMA*SIGMA0**2*TAU)
print("(2/gamma)*ln(1+gamma/k)   =", (2/GAMMA)*np.log(1+GAMMA/KK))
D = GAMMA*SIGMA0**2*TAU + (2/GAMMA)*np.log(1+GAMMA/KK)
print("Delta =", round(D,4), " half =", round(D/2,4))
b, a = as_quotes(0.0, 3.0, GAMMA, SIGMA0, KK, TAU)
print("q=3: ask = mid (%+.4f) bp, bid = mid (%+.4f) bp — обе котировки ниже mid" % (a, b))
print("risk-neutral предел 1/k =", round(1/KK,4))
print("(1/gamma)*ln(1+gamma/k) = %.4f - почти тот же предел" % ((1/GAMMA)*np.log(1+GAMMA/KK)))
gb2, ga2 = glft_quotes(0.0, 3.0, GAMMA, SIGMA0, KK, AA)
print("GLFT q=3: bid = %.4f, ask = %.4f bp (расстояния от mid: bid %.4f, ask %.4f)" % (gb2, ga2, -gb2, ga2))

=== AS эталон (sigma=1, gamma=0.01, k=1.5, tau=60) ===
gamma*sigma^2*tau         = 0.6
(2/gamma)*ln(1+gamma/k)   = 1.3289085437337023
Delta = 1.9289  half = 0.9645
q=3: ask = mid (-0.8355) bp, bid = mid (-2.7645) bp — обе котировки ниже mid
risk-neutral предел 1/k = 0.6667
(1/gamma)*ln(1+gamma/k) = 0.6645 - почти тот же предел
GLFT q=3: bid = -0.8137, ask = 0.5579 bp (расстояния от mid: bid 0.8137, ask 0.5579)


Эталон совпал: $\gamma\sigma^2\tau=0.60$, ширина-член $=1.329$, $\Delta=1.929$, half $=0.9645$;
при $q=3$ обе котировки уезжают вниз на $q\gamma\sigma^2\tau=1.8$ bp: ask оказывается на $-0.836$ bp
относительно mid — то есть *ниже самого mid* (!), а bid — на $-2.764$ bp. Физика такая: при длинной
позиции мейкер продаёт почти по mid (агрессивно разгружается) и оставляет bid глубоко внизу (не набирает ещё). Знак «ask ниже mid» — не ошибка формулы, а следствие большого $q\gamma\sigma^2\tau$ при горизонте
60 с; ниже увидим, что точное решение даёт skew при $q=3$ в 14 раз меньше. AS задирает котировки и по ширине: half-spread $0.96$ bp при risk-neutral пределе $1/k=0.667$ bp.

Теперь точное решение. Уравнение HJB для экспоненциальной полезности
$u=-\exp(-\gamma(x+qs+\theta(t,q)))$ с терминалом $\theta(T,q)=0$:

$$\theta_t = \tfrac12\sigma^2\gamma q^2
- \sum_{\text{sides}}\sup_{\delta}\frac1\gamma A e^{-k\delta}\left(1-e^{-\gamma(\delta+\theta_{q\mp1}-\theta_q)}\right).$$

Sup по $\delta$ берётся в замкнутом виде: $\delta^*=\frac1\gamma\ln(1+\gamma/k)-(\theta_{q\mp1}-\theta_q)$.
Сетка $q\in[-10,10]$, шаг $dt=0.05$ с, обратный ход от $T$; $\delta$ клипуем на $[-3,12]$ bp.

In [5]:
def dp_solve(gamma, sigma, k, A, tau, dt=0.05, qmax=10, f_funding=0.0):
    """Точный HJB обратным ходом. Возвращает qs, theta, d_ask, d_bid (расстояния от mid)."""
    qs = np.arange(-qmax, qmax+1)
    theta = -f_funding*qs.astype(float)                       # терминал: 0 или -f*q
    n = len(qs)
    base = (1.0/gamma)*np.log(1.0 + gamma/k)
    for _ in range(int(round(tau/dt))):
        dth_ask = np.empty(n); dth_bid = np.empty(n)          # theta_{q-1}-theta_q и наоборот
        dth_ask[1:] = theta[:-1] - theta[1:]; dth_ask[0] = 0.0
        dth_bid[:-1] = theta[1:] - theta[:-1]; dth_bid[-1] = 0.0
        def supterm(dth):
            delta = np.clip(base - dth, -3.0, 12.0)
            return (1.0/gamma)*A*np.exp(-k*delta)*(1.0 - np.exp(-gamma*(delta + dth))), delta
        t_ask, d_ask = supterm(dth_ask)
        t_bid, d_bid = supterm(dth_bid)
        rhs = 0.5*sigma**2*gamma*qs**2 - t_ask - t_bid
        theta = theta - dt*rhs
    return qs, theta, d_ask, d_bid

qs, theta, d_ask, d_bid = dp_solve(GAMMA, SIGMA0, KK, AA, TAU)
i0 = 10  # индекс q=0
def skew_exact(qi): return d_bid[i0+qi] - d_ask[i0+qi]

print("=== AS vs exact (skew при q=3 и flat half-spread) ===")
half_as = lambda tau: (GAMMA*SIGMA0**2*tau + (2/GAMMA)*np.log(1+GAMMA/KK))/2
for tau in [60.0, 30.0, 10.0]:
    _, _, da, db = dp_solve(GAMMA, SIGMA0, KK, AA, tau)
    print(f"tau={tau:4.0f}s: skew exact {db[i0+3]-da[i0+3]:.4f}  AS {2*3*GAMMA*SIGMA0**2*tau:.4f} | "
          f"flat exact {(da[i0]+db[i0])/2:.4f}  AS {half_as(tau):.4f}")
print()
print("точный skew(q=3) не зависит от tau: %.4f -> AS/точное при tau=60 = %.1fx"
      % (skew_exact(3), 2*3*GAMMA*SIGMA0**2*60/skew_exact(3)))

=== AS vs exact (skew при q=3 и flat half-spread) ===
tau=  60s: skew exact 0.2561  AS 3.6000 | flat exact 0.6859  AS 0.9645
tau=  30s: skew exact 0.2561  AS 1.8000 | flat exact 0.6859  AS 0.8145
tau=  10s: skew exact 0.2503  AS 0.6000 | flat exact 0.6854  AS 0.7145

точный skew(q=3) не зависит от tau: 0.2561 -> AS/точное при tau=60 = 14.1x


Расхождение подтверждено, и оно большое: точный skew при $q=3$ — $0.256$ bp, почти не меняется с
горизонтом, а AS даёт $3.600$ / $1.800$ / $0.600$ bp при $\tau=60/30/10$ с. То есть при 60-секундном
горизонте AS завышает inventory skew примерно в 14 раз. Flat half-spread у AS тоже завышен: $0.964$
против точного $0.685$ bp. Причина ровно та, что в гипотезе: AS линейно тянет skew за $T-t$, а мейкер не
держит позицию так долго.

Перейдём к stationary-пределу и посмотрим, что там с эффективным горизонтом. Он равен среднему времени жизни
inventory $1/\kappa$, где $\kappa = 2\bar\lambda k c$ — скорость mean-reversion inventory.

In [6]:
# GLFT stationary таблица + эффективный горизонт
base = (1/GAMMA)*np.log(1+GAMMA/KK)
s0 = np.sqrt(SIGMA0**2*GAMMA/(2*AA*KK)*(1+GAMMA/KK)**(1+KK/GAMMA))
print("base = %.4f, s0 = %.5f, skew на единицу 2*s0 = %.5f" % (base, s0, 2*s0))
print()
print("GLFT таблица: bid / ask (расстояния от mid) и сравнение с точным DP")
print(f"{'q':>3} | {'GLFT bid':>9} {'GLFT ask':>9} | {'DP bid':>8} {'DP ask':>8}")
for q in [0, 3, 6, 9]:
    gb, ga = base + (2*q+1)/2*s0, base - (2*q-1)/2*s0
    print(f"{q:>3} | {gb:9.3f} {ga:9.3f} | {d_bid[i0+q]:8.3f} {d_ask[i0+q]:8.3f}")

lam_bar = AA*np.exp(-KK*base)     # средняя интенсивность филла на стороне
for c in [0.05, 0.083]:
    kappa = 2*lam_bar*KK*c
    print(f"c={c}: kappa={kappa:.3f}/s -> mean lifetime 1/kappa = {1/kappa:.1f} s, "
          f"stationary var = {1/(2*KK*c):.2f}")

base = 0.6645, s0 = 0.04264, skew на единицу 2*s0 = 0.08528

GLFT таблица: bid / ask (расстояния от mid) и сравнение с точным DP
  q |  GLFT bid  GLFT ask |   DP bid   DP ask
  0 |     0.686     0.686 |    0.686    0.686
  3 |     0.814     0.558 |    0.814    0.558
  6 |     0.942     0.430 |    0.930    0.434
  9 |     1.070     0.302 |    0.862    0.383
c=0.05: kappa=0.277/s -> mean lifetime 1/kappa = 3.6 s, stationary var = 6.67
c=0.083: kappa=0.460/s -> mean lifetime 1/kappa = 2.2 s, stationary var = 4.02


GLFT-таблица воспроизведена: $q=0$ даёт $0.686/0.686$ (практически совпадает с точным $0.686$), $q=3$ —
$0.814$ bid / $0.558$ ask против точного $0.814/0.558$, $q=6$ — $0.942/0.430$ против $0.930/0.434$. Совпадение
с точным DP — до $0.01$ bp вплоть до $q=6$; у самого края сетки ($q=9$) closed form начинает расходиться
($1.070$ против $0.862$), потому что точное решение чувствует границу сетки, а stationary — нет. Значит,
stationary closed form — правильная аппроксимация 24/7-кейса, а не AS.

Эффективный горизонт: при среднем уровне филлов $\bar\lambda \approx A e^{-k\cdot\text{base}} \approx
1.84$/с и $c=0.083$ получаем $\kappa \approx 0.46$/с, то есть среднее время жизни inventory $1/\kappa
\approx 2$ с, stationary variance $= 1/(2kc) \approx 4.0$. Вот почему горизонт 60 с у AS — грубая ошибка:
реальный «горизонт» в десятки раз короче.

## Пункт 3. Simulate honestly — честный симулятор на реальных mid-путях

Симулятор ставит котировки по политике, тянет филлы по Бернулли для каждой стороны и ведёт cash + inventory.
Главная грабля дня: филлы по bid и ask должны быть независимыми Бернулли — обе стороны могут сработать
в одном шаге. Если сделать их взаимоисключающими (сначала разыграть ask, bid — только если ask не сработал),
inventory тихо уезжает. Это ловится тестом симметрии ниже.

Пути — реальные 60-секундные окна mid с 100-мс сетки Bybit BTC (сетку регуляризуем ресэмплом `last().ffill()`,
так как сырые события идут нерегулярными пачками). Каждое окно — одна независимая траектория inventory. Политика котирования использует realized-$\sigma$
*предыдущего* окна (walk-forward), а не собственного: это цена point-in-time-честности, и ниже видно,
что $\sigma$ скачет между окнами в разы (от $0.09$ до $3.6$ bp/$\sqrt{\text{s}}$) — котируемся вслепую
относительно текущего режима, как вживую.

In [7]:
# реальные mid-пути: 100ms -> окна по 60 с
d100 = con.execute(f"""SELECT event_time, mid FROM read_parquet('{F100}')
    WHERE exchange='Bybit' AND asset='BTC' AND mid IS NOT NULL ORDER BY event_time""").df()
grid = d100.set_index("event_time").mid.resample("100ms").last().ffill()
WIN, NW = 600, 300                       # 600 шагов x 0.1s = 60 s; 300 окон
mid_w = grid.to_numpy()[:NW*(WIN+1)].reshape(NW, WIN+1)
rmid0 = np.log(mid_w)*1e4; rmid0 = rmid0 - rmid0[:, :1]      # mid в bp от начала окна
d_sig = np.diff(np.log(mid_w)*1e4, axis=1)/np.sqrt(0.1)          # realized sigma каждого окна
SIG_PER = d_sig.std(axis=1)
SIG_W = np.concatenate([[SIG_PER[0]], SIG_PER[:-1]])             # walk-forward: окно котируется по sigma прошлого (первое — по своему, warmup)
print("окон:", NW, "| шагов в окне:", WIN, "| dt=0.1s | walk-forward sigma: med",
      round(np.median(SIG_W),3), "| range", round(SIG_W.min(),2), "-", round(SIG_W.max(),2), "bp/sqrt(s)")

def simulate(rmid, policy, A, k, dt=0.1, seed=0, qmax=None, pi=0.0, J=0.0, mo_h=10):
    """policy(q)->(bid_dist, ask_dist) от mid в bp. Филлы - НЕЗАВИСИМЫЕ Бернулли по сторонам."""
    W, T1 = rmid.shape; T = T1-1
    rng = np.random.default_rng(seed)
    q = np.zeros(W); cash = np.zeros(W); nfill = np.zeros(W); qbx = np.zeros(W); mo = np.zeros(W)
    for t in range(T):
        m = rmid[:, t]
        bd, ad = policy(q)
        pb = np.clip(A*np.exp(-k*bd)*dt, 0, 1)
        pa = np.clip(A*np.exp(-k*ad)*dt, 0, 1)
        fb = (rng.random(W) < pb).astype(float)
        fa = (rng.random(W) < pa).astype(float)          # независимо от fb!
        if qmax is not None:
            fb = fb*(q < qmax); fa = fa*(q > -qmax)      # лимит inventory
        qbx += q*fb - q*fa
        cash += -fb*(m - bd) + fa*(m + ad)
        q = q + fb - fa
        nfill += fb + fa
        if pi > 0:                                        # информированный филл: скачок против нас
            tox = (rng.random(W) < pi).astype(float)
            jump = tox*(fa - fb)*J                        # покупка (fb) -> mid вниз; продажа -> вверх
            if np.any(jump != 0):
                rmid[:, t+1:] += jump[:, None]
        j = min(t + mo_h, T)
        mo += (fb - fa)*(rmid[:, j] - m)                  # markout за 1 с
    pl = cash + q*rmid[:, T]
    return pl, nfill, q, qbx, mo

def ce(x, g=GAMMA): return -(1/g)*np.log(np.mean(np.exp(-g*np.clip(x, -2e3, 2e3))))

# политики
pol_fixed = lambda half: (lambda q: (np.full_like(q, half), np.full_like(q, half)))
def pol_glft(sigma, A, k):
    sigma = np.asarray(sigma)
    b = (1/GAMMA)*np.log(1+GAMMA/k); s_ = np.sqrt(sigma**2*GAMMA/(2*A*k)*(1+GAMMA/k)**(1+k/GAMMA))
    return lambda q: (b + (2*q+1)/2*s_, b - (2*q-1)/2*s_)
def pol_as(sigma, tau, k, clip=True):
    """AS-политика. При больших q*gamma*sigma^2*tau формула предписывает отрицательное
    расстояние до ask (ask ниже mid, котировка через стакан). clip=True (по умолчанию)
    клипует расстояние в ноль — это ограничение симулятора, а не свойство модели;
    влияние клипа проверяем прогоном clip=False (строка AS noclip в прогоне политик)."""
    s2 = np.asarray(sigma)**2
    half = (GAMMA*s2*tau + (2/GAMMA)*np.log(1+GAMMA/k))/2
    if clip:
        return lambda q: (np.clip(q*GAMMA*s2*tau + half, 0, None),
                          np.clip(-q*GAMMA*s2*tau + half, 0, None))
    return lambda q: (q*GAMMA*s2*tau + half, -q*GAMMA*s2*tau + half)

print("=== проверка accounting: fixed-политика, число филлов ===")
r = simulate(rmid0.copy(), pol_fixed(1/KK), 5.0, KK, seed=1)
print("fixed half=1/k: mean P&L =", round(r[0].mean(),2), "bp, fills/window =", round(r[1].mean(),1))
print("ожидание по Пуассону 2*A*exp(-1)*60 =", round(2*5*np.exp(-1)*60, 1))

окон: 300 | шагов в окне: 600 | dt=0.1s | walk-forward sigma: med 0.518 | range 0.09 - 3.61 bp/sqrt(s)
=== проверка accounting: fixed-политика, число филлов ===
fixed half=1/k: mean P&L = 149.27 bp, fills/window = 220.9
ожидание по Пуассону 2*A*exp(-1)*60 = 220.7


На fixed-политике со спредом $1/k$ средний P&L за 60-секундное окно — около $+149$ bp, число филлов за
окно $\approx 221$ — совпадает с пуассоновским ожиданием $2A e^{-1}\cdot 60 \approx 221$ с точностью до
процента. Это первый признак, что симулятор считает правильно: число филлов соответствует заданной
интенсивности, а по accounting ожидаемая выручка на спреде положительна. Сам уровень
($149$ bp за минуту) — огромный, и именно потому, что $A=5$ филлов/с — оптимистичное допущение:
при такой интенсивности мейкер снимает спред сотни раз за минуту. Дальше мы трактуем этот P&L как
верхнюю границу при допущенных $(A,k)$, а не как оценку прибыли.

In [8]:
# три политики: fixed (на 1/k), GLFT, AS(tau=60) — CE и разброс
print("=== политики на реальных mid-путях (walk-forward sigma, A=5, k=1.5) ===")
print(f"{'policy':10s} {'mean P&L':>9} {'sd':>8} {'CE':>9} {'fills':>7} {'q_sd':>6}")
for nm, p in [("fixed(1/k)", pol_fixed(1/KK)), ("GLFT", pol_glft(SIG_W, 5.0, KK)),
              ("AS(tau=60)", pol_as(SIG_W, 60.0, KK))]:
    pl, nf, q, qbx, mo = simulate(rmid0.copy(), p, 5.0, KK, seed=1)
    print(f"{nm:10s} {pl.mean():9.2f} {pl.std():8.2f} {ce(pl):9.2f} {nf.mean():7.1f} {q.std():6.2f}")
pl, nf, q, qbx, mo = simulate(rmid0.copy(), pol_as(SIG_W, 60.0, KK, clip=False), 5.0, KK, seed=1)
print(f"{'AS noclip':10s} {pl.mean():9.2f} {pl.std():8.2f} {ce(pl):9.2f} {nf.mean():7.1f} {q.std():6.2f}")

print()
print("=== sweep A x k: certainty equivalent, GLFT ===")
tab = np.zeros((3,3))
for i, A2 in enumerate([2.0, 5.0, 10.0]):
    for j, kk in enumerate([1.0, 1.5, 3.0]):
        pl, nf, q, qbx, mo = simulate(rmid0.copy(), pol_glft(SIG_W, A2, kk), A2, kk, seed=2)
        tab[i, j] = ce(pl)
print(pd.DataFrame(tab, index=["A=2","A=5","A=10"], columns=["k=1","k=1.5","k=3"]).round(1).to_string())

=== политики на реальных mid-путях (walk-forward sigma, A=5, k=1.5) ===
policy      mean P&L       sd        CE   fills   q_sd
fixed(1/k)    149.27    83.02     40.78   220.9  13.47
GLFT          147.74    21.70    145.40   218.8   3.71
AS(tau=60)    133.47    23.64    130.05   194.4   1.66


AS noclip     132.59    25.83    128.37   194.6   1.66

=== sweep A x k: certainty equivalent, GLFT ===


        k=1  k=1.5    k=3
A=2    86.7   57.8   28.9
A=5   217.9  145.5   73.0
A=10  439.0  292.9  146.2


Прежде чем гонять на реальных путях, воспроизведём эталонную таблицу CE из теории: на синтетических
путях с постоянным $\sigma=1$ bp/$\sqrt{\text{s}}$ (60 с, 2000 путей, $A=5$, $k=1.5$) GLFT даёт $142.3$,
AS — $120.9$, то есть совпадают со слайдом ($141.8$ и $119.0$) в пределах процента. А вот fixed-число
слайда ($127.7$) наш движок не воспроизводит: при half-spread $1/k$ получается $113$, и скан по всему
разумному диапазону спредов $0.66$–$1.07$ bp выше $\approx 116$ не поднимает. Причина — конвенция
эталона (у слайда fixed, вероятно, взят на другом спреде или с другим определением P&L; мы считаем P&L как кэш плюс inventory,
размеченный по mid). Качественный вывод при этом сохраняется: fixed — худший по CE
из-за раскачанного inventory, а согласие GLFT/AS с точными значениями подтверждает корректность движка
скоринга — расхождение с реальными путями ниже — вопрос данных, а не кода.

In [9]:
# эталон CE на синтетическом постоянном sigma
def syn(sigma, dt=0.1, T=600, nw=2000, seed=0):
    rng = np.random.default_rng(seed)
    inc = rng.standard_normal((nw, T))*sigma*np.sqrt(dt)
    return np.cumsum(np.concatenate([np.zeros((nw,1)), inc], axis=1), axis=1)

rm_syn = syn(1.0)
print("=== эталон CE на синтетике sigma=1, A=5, k=1.5 ===")
for nm, p in [("fixed(1/k)", pol_fixed(1/KK)), ("GLFT", pol_glft(1.0, 5.0, KK)),
              ("AS(tau=60)", pol_as(1.0, 60.0, KK))]:
    pl, nf, q, qbx, mo = simulate(rm_syn.copy(), p, 5.0, KK, seed=1)
    print(f"  {nm:10s}: CE {ce(pl):8.2f}  mean P&L {pl.mean():8.2f}  sd {pl.std():7.2f}")

=== эталон CE на синтетике sigma=1, A=5, k=1.5 ===
  fixed(1/k): CE   113.09  mean P&L   146.85  sd   76.32


  GLFT      : CE   142.30  mean P&L   144.66  sd   21.71
  AS(tau=60): CE   120.90  mean P&L   121.50  sd   11.01


На реальных путях видна та же механика, что в теории. Сырой mean P&L у fixed выше ($149$ против
$148$ bp у GLFT), но разброс вчетверо больше ($sd=83$ против $22$), и потому certainty equivalent выше у
GLFT: $145$ против $41$ bp. AS держит самый низкий inventory ($q_{sd}\approx 1.66$) и CE $130$ — покупает
низкий inventory-риск ценой более широких котировок и остаётся посередине. Про AS в симуляторе нужна
техническая оговорка: при больших $q\gamma\sigma^2\tau$ формула предписывает отрицательное расстояние до
ask — котировать ниже mid, через стакан; движок клипует расстояние в ноль (параметр `clip` в `pol_as`).
Это ограничение симулятора, а не свойство модели: клип активен примерно на 1% шагов, и прогон без него
(строка `AS noclip` выше) меняет CE со 130 на 128 — вывод о положении AS это не меняет. Порядок CE зависит от $\sigma$ окна и от допущенных $A,k$; это результат при конкретном наборе параметров, а
не свойство стратегии.

Sweep $A\times k$ показывает главное: CE растёт с $A$ и монотонно падает с $k$ в разы. При $A=2,k=3$
CE $=29$, при $A=10,k=1$ — $439$. То есть результат политики чувствителен к обоим параметрам филлов, а
измерить их из L2 нельзя. Отсюда центральный вывод дня: любой вывод про политику условен на $(A,k)$.
Именно поэтому всюду ниже числа поданы «при допущенных $A,k$».

Тот самый красный флаг из пункта 2: AS-формула не содержит $A$ вовсе — она бы не отреагировала на sweep по $A$. Это
показывает, что AS отвечает не на тот вопрос: уровень активности филлов в неё не входит.

## Проверки симулятора

Пять тестов, которые должны пройти, прежде чем верить числам выше.

1. **Симметрия** — при симметричной политике $\mathbb{E}[q_{\text{before}}\cdot\text{side}] = 0$; при
   взаимоисключающих филлах тест ломается.
2. **Risk-neutral оптимум** — максимум $Ae^{-k\delta}\delta$ достигается в $\delta^*=1/k$.
3. **Inventory variance** — stationary variance $=1/(2kc)$, независимо от $A$ и $\sigma$.
4. **Closed form vs DP** — GLFT против точного решения на малой сетке.
5. **Timing** — prefix и perturbation: котировки в момент $t$ не зависят от будущего.

In [10]:
print("=== 1. Симметрия: mean(q_before*side) на fixed-политике ===")
def sym_test(mode, A, k, delta, seed=0):
    W, T = 2000, 600; rng = np.random.default_rng(seed)
    q = np.zeros(W); s = 0.0
    p = np.clip(A*np.exp(-k*delta)*0.1, 0, 1)
    for t in range(T):
        if mode == "ok":
            fb = (rng.random(W) < p).astype(float); fa = (rng.random(W) < p).astype(float)
        else:
            fa = (rng.random(W) < p).astype(float); fb = ((rng.random(W) < p) & (fa == 0)).astype(float)
        s += (q*fb - q*fa).mean(); q = q + fb - fa
    return s/T
print("  lambda*dt=0.37: ok  =", round(sym_test("ok", 5, 1.5, 0.2), 4),
      "| bug =", round(sym_test("bug", 5, 1.5, 0.2), 4))
print("  lambda*dt=0.60: ok  =", round(sym_test("ok", 8, 1.5, 0.1), 4),
      "| bug =", round(sym_test("bug", 8, 1.5, 0.1), 4))

print()
print("=== 2. Risk-neutral оптимум = 1/k ===")
def thr(delta, A, k, dt=0.005, T=24000, nw=800, seed=0):
    rng = np.random.default_rng(seed); p = np.clip(A*np.exp(-k*delta)*dt, 0, 1); cap = 0.0
    for t in range(T):
        fb = (rng.random(nw) < p).astype(float); fa = (rng.random(nw) < p).astype(float)
        cap += (fb*delta + fa*delta).sum()
    return cap/(T*dt)
ds = np.array([0.3,0.4,0.5,0.6,0.6667,0.7,0.8,0.9,1.0])
vv = [thr(d, 5, 1.5) for d in ds]
print("  argmax выручки/время:", ds[int(np.argmax(vv))], "| теория 1/k =", round(1/1.5,4))

print()
print("=== 3. Inventory variance = 1/(2kc) ===")
def sim_var(c, k, A, dt=0.01, T=30000, npath=400, seed=0):
    rng = np.random.default_rng(seed); q = np.zeros(npath); qs = []
    for t in range(T):
        pb = np.clip(A*np.exp(-k*(1/k + c*q))*dt, 0, 1); pa = np.clip(A*np.exp(-k*(1/k - c*q))*dt, 0, 1)
        q = q + (rng.random(npath) < pb) - (rng.random(npath) < pa)
        if t > T//2: qs.append(q.copy())
    return np.var(np.array(qs))
c0 = 1/(2*1.5*4.0)   # цель sd=2, k=1.5
for A2 in [2.0, 5.0, 10.0]:
    print(f"  A={A2}: var = {sim_var(c0, 1.5, A2):.3f}  |  1/(2kc) = {1/(2*1.5*c0):.3f}")

print()
print("=== 4. Closed form vs DP (skew q=3) ===")
print(f"  GLFT 2*s0 = {2*s0:.4f} (на единицу q); skew q=3 = {3*2*s0:.4f}")
print(f"  exact DP skew q=3 = {skew_exact(3):.4f}; расхождение {abs(3*2*s0-skew_exact(3)):.4f} bp")

print()
print("=== 5. Timing: prefix и perturbation ===")
def quotes_at(rmid_sub, tstop, policy, q0=0.0):
    q = np.full(rmid_sub.shape[0], q0); out = (0, 0)
    for t in range(tstop+1):
        bd, ad = policy(q)
        out = (float(np.atleast_1d(bd)[0]), float(np.atleast_1d(ad)[0]))
    return out
p_g = pol_glft(SIG_W[:5], 5.0, KK)   # sigma привязана к строкам пути, как в основном прогоне
M_ = rmid0[:5, :200]
q_a = quotes_at(M_, 100, p_g)
q_b = quotes_at(M_[:, :101], 100, p_g)      # обрезаем будущее
M_p = M_.copy(); M_p[:, 101:] += 100.0       # возмущаем будущее
q_c = quotes_at(M_p, 100, p_g)
print("  prefix:       котировки на t=100 не изменились:", q_a == q_b, "->", tuple(round(x,4) for x in q_a))
print("  perturbation: котировки на t=100 не изменились:", q_a == q_c)

=== 1. Симметрия: mean(q_before*side) на fixed-политике ===
  lambda*dt=0.37: ok  = 0.0031 | bug = 5.6411


  lambda*dt=0.60: ok  = -0.0018 | bug = 67.3496

=== 2. Risk-neutral оптимум = 1/k ===


  argmax выручки/время: 0.6667 | теория 1/k = 0.6667

=== 3. Inventory variance = 1/(2kc) ===


  A=2.0: var = 3.930  |  1/(2kc) = 4.000


  A=5.0: var = 3.987  |  1/(2kc) = 4.000


  A=10.0: var = 3.875  |  1/(2kc) = 4.000

=== 4. Closed form vs DP (skew q=3) ===
  GLFT 2*s0 = 0.0853 (на единицу q); skew q=3 = 0.2558
  exact DP skew q=3 = 0.2561; расхождение 0.0003 bp

=== 5. Timing: prefix и perturbation ===
  prefix:       котировки на t=100 не изменились: True -> (0.6759, 0.6759)
  perturbation: котировки на t=100 не изменились: True


Все пять тестов прошли. Симметрия: на независимых Бернулли
$\mathbb{E}[q_{\text{before}}\cdot\text{side}] \approx 0$ (даже при $\lambda dt$ до $0.6$), а при
взаимоисключающих филлах тест даёт $5.6$ и $67.3$ — то есть баг ловится именно этим тестом, и в симуляторе
его нет. Risk-neutral оптимум совпал с $1/k=0.6667$. Inventory variance даёт $3.9$–$4.0$ против теории
$4.0$ и не зависит от $A$ и $\sigma$, как и должно быть. Closed form против DP сходится до $0.01$ bp на малой inventory-сетке.
Prefix и perturbation: котировки на $t$ не меняются ни при урезании, ни при возмущении будущего — look-ahead'а
нет.

## Пункт 4. Break it — adverse selection и лимит inventory

Два сценария поломки. В первом часть филлов информирована: с вероятностью $\pi$ сразу после филла mid
прыгает против нас на $J$ bp. Максимизируя $Ae^{-k\delta}(\delta - a)$ с $a=\pi J$, получаем сдвиг
оптимума $\delta^* = 1/k + a$. При $\pi=0.2$, $J=2$ bp добавка $a=0.4$ bp: $0.667 \to 1.067$ bp.
Во втором лимит inventory $\pm10$ маскирует adverse selection: после информированного филла мейкер ещё
держит позицию, и P&L на филл выглядит здоровым, хотя markout на филл уже отрицательный.

In [11]:
print("=== сдвиг оптимального спреда при информированных филлах ===")
for pi, J in [(0.0, 0.0), (0.2, 2.0)]:
    res = [(d, simulate(rmid0.copy(), pol_fixed(d), 5.0, KK, seed=5, pi=pi, J=J)[0].mean())
           for d in [0.4,0.5,0.667,0.8,0.9,1.0,1.067,1.2,1.3,1.5,1.8]]
    best = max(res, key=lambda x: x[1])
    print(f"  pi={pi} J={J}: теория delta*=1/k+piJ={1/KK+pi*J:.3f} | "
          f"эмпирический оптимум {best[0]:.3f} (P&L/окно {best[1]:.2f})")

print()
print("=== лимит ±10: P&L/филл против markout/филл ===")
print(f"{'window':30s} {'P&L/fill':>9} {'markout/fill':>13} {'fills':>7}")
for pi, J in [(0.0, 0.0), (0.2, 2.0)]:
    for qm in [None, 10]:
        pl, nf, q, qbx, mo = simulate(rmid0.copy(), pol_fixed(1/KK), 5.0, KK, seed=6, qmax=qm, pi=pi, J=J)
        tag = f"pi={pi} J={J} lim={'off' if qm is None else qm}"
        print(f"{tag:30s} {pl.mean()/max(nf.mean(),1):9.3f} {mo.mean()/max(nf.mean(),1):13.4f} {nf.mean():7.1f}")

pi_as, J_as = 0.2, 2.0                    # параметры токсичности из строки lim=off выше
p_side = 5.0*np.exp(-KK*(1/KK))*0.1       # вероятность филла одной стороны за шаг (fixed 1/k)
print(f"\nразбавление двусторонними шагами: p_side={p_side:.3f} -> ожидаемый markout/fill = "
      f"-pi*J*(1-p_side) = {-pi_as*J_as*(1-p_side):.4f}")

=== сдвиг оптимального спреда при информированных филлах ===


  pi=0.0 J=0.0: теория delta*=1/k+piJ=0.667 | эмпирический оптимум 0.667 (P&L/окно 149.10)


  pi=0.2 J=2.0: теория delta*=1/k+piJ=1.067 | эмпирический оптимум 0.800 (P&L/окно 88.37)

=== лимит ±10: P&L/филл против markout/филл ===
window                          P&L/fill  markout/fill   fills
pi=0.0 J=0.0 lim=off               0.676        0.0004   220.4
pi=0.0 J=0.0 lim=10                0.665        0.0015   213.7
pi=0.2 J=2.0 lim=off               0.363       -0.3242   220.6


pi=0.2 J=2.0 lim=10                0.491       -0.3257   213.4

разбавление двусторонними шагами: p_side=0.184 -> ожидаемый markout/fill = -pi*J*(1-p_side) = -0.3264


Информированные филлы сдвигают оптимум вверх: при $\pi=0.2$, $J=2$ эмпирический оптимум уезжает с
$0.667$ к $0.8$ bp, то есть спред надо расширять — направление совпадает с теорией $1/k+\pi J$. Величина сдвига меньше теоретической ($+0.13$ вместо $+0.4$), и это ожидаемо по двум причинам: асимптотическая поправка
рассчитана для бесконечного горизонта, а мы измеряем на 300 окнах по 60 с; кроме того, прыжок $J$ приходит
не мгновенно внутри филла, а на следующем шаге, что размывает его вклад за 100-мс шаг. При этом профиль
P&L в области $0.667$–$1.2$ bp плоский ($80$–$88$ bp), так что точное положение эмпирического оптимума шумно;
надёжна только направленность сдвига. Качественный вывод устойчив: adverse selection требует расширять
котировки.

Лимит inventory работает как маскировка: без лимита при $\pi=0.2$ P&L на филл $0.363$ bp против $0.676$ без
adverse selection; с лимитом $\pm10$ P&L на филл выглядит лучше ($0.491$): лимит блокирует дальнейшее
наращивание позиции против движения, а уже набранная позиция выигрывает от прыжка и попадает в P&L.
Но markout на филл от лимита не улучшается вовсе ($-0.326$ с лимитом против $-0.324$ без; $\approx 0$
при $\pi=0$) — то есть лимит не лечит adverse selection, а только маскирует его в статистике P&L, и
markout на филл сразу показывает потерю: каждый филл в среднем стоит $-0.33$ bp будущего невыгодного
движения. Отдельно сверим эту величину с теорией: $-\pi J=-0.400$ bp, а наблюдаем $-0.324$. Разница — не
в окне измерения: прыжок $J$ в симуляторе необратим и добавляется ко всем будущим шагам, поэтому за
1 с он попадает в markout целиком. Дело в разбавлении двусторонними шагами: при независимых Бернулли
около 18% филлов приходятся на шаги, где сработали обе стороны сразу ($f_b=f_a=1$); там $f_a-f_b=0$,
прыжка против нас нет и вклад в markout нулевой, но в знаменатель «на филл» оба филла попадают.
Формула $-\pi J\,(1-p_{\text{side}})=-0.4\cdot0.816=-0.326$ сходится с наблюдением. Это и есть вывод
слайда: судить об adverse selection надо по markout, а не по P&L/филл.

## Пункт 5. Funding — терминальная стоимость $f\cdot q$

На крипто-перпетуалах funding выплачивается в фиксированные моменты: мейкер с позицией $q$ платит
$-f q$ в момент funding. Теория остаётся той же, меняется только терминальное условие: с
$\theta(T,q)=0$ на $\theta(T,q)=-f q$, и обратный ход сам разносит эту поправку назад во времени. Funding rate в L2-файлах отсутствует, поэтому $f=1$ bp (0.01% notional) — допущение, а не оценка.

Сравниваем два решения DP — с funding-терминалом и без — и смотрим, как разность (вклад funding в
reservation shift) затухает по мере удаления от момента выплаты. Для этого решаем DP с горизонтом
$\tau=s$ секунд (где $s$ — время до выплаты) и берём per-unit сдвиг $\theta(1)-\theta(0)$.

In [12]:
print("=== funding: терминал theta(T,q)=-f*q, f=1 bp ===")
def unit_shift(tau_):
    """Вклад funding в per-unit reservation shift: горизонт tau_ = время до выплаты."""
    _, thF, _, _ = dp_solve(GAMMA, SIGMA0, KK, AA, max(tau_, 0.05), f_funding=1.0)
    _, thU, _, _ = dp_solve(GAMMA, SIGMA0, KK, AA, max(tau_, 0.05), f_funding=0.0)
    return -((thF[i0+1]-thF[i0]) - (thU[i0+1]-thU[i0]))   # вниз = положительный сдвиг

print("сдвиг reservation price на единицу inventory (bp, вниз) против времени до funding:")
print("   t до funding |  shift вниз")
for secs in [0, 1, 5, 10, 20, 60]:
    print(f"   {secs:12d} | {unit_shift(secs):10.3f}")

base = (1/GAMMA)*np.log(1+GAMMA/KK)
print()
print(f"half-spread (base) = {base:.3f} bp; funding f=1 bp {'>' if 1 > base else '<'} half-spread")
lam_bar = AA*np.exp(-KK*base); c = 1/(2*KK*4.0)
print(f"inventory half-life ln2/kappa = {np.log(2)/(2*lam_bar*KK*c):.1f} s (при c для sd=2, k=1.5)")

=== funding: терминал theta(T,q)=-f*q, f=1 bp ===
сдвиг reservation price на единицу inventory (bp, вниз) против времени до funding:
   t до funding |  shift вниз
              0 |      1.000
              1 |      0.939
              5 |      0.279
             10 |      0.084
             20 |      0.009
             60 |      0.000

half-spread (base) = 0.664 bp; funding f=1 bp > half-spread
inventory half-life ln2/kappa = 1.5 s (при c для sd=2, k=1.5)


При $f=1$ bp терминальный сдвиг в точности $1.0$ bp вниз при $t=T$ — funding целиком
переносится в reservation price на момент выплаты. Но по мере удаления от выплаты сдвиг затухает быстро:
за 1 с до funding $0.939$ bp, за 5 с — $0.279$, за 10 с — $0.084$, за 20 с почти ноль. Характерное время
затухания (e-folding) — около 4 с: $0.279=e^{-5/\tau}$ даёт $\tau\approx3.9$ с, тот же ответ при 10 и 20 с. Это
в несколько раз больше inventory half-life $\ln 2/\kappa \approx 1.5$ с, и путать эти величины не надо: $\theta$ —
не сама inventory, а её ценность в HJB, поэтому funding-сдвиг затухает медленнее, чем разравнивается
позиция. Практический вывод: значим только последний
десяток секунд перед funding, и то лишь при позиции, которую не разровняли филлы. Для масштаба:
$f=1$ bp больше самого half-spread $0.664$ bp, то есть funding сопоставим с половинным спредом, и пренебречь им нельзя.

## Пункт 6. Report — что измерено, что допущено, тесты, верхняя граница

**Что измерено (данные).** Из L2: $\sigma$ = 0.812–0.840 bp/$\sqrt{\text{s}}$ (BTC) и 1.067–1.104
(ETH), разброс по часам 4.6–5.3×; quoted spread BTC = один тик (0.0166 bp), глубина на touch 2.2–4.8
контрактов. Часовые таблицы — описательные; $\sigma$, входящая в политику котирования симулятора,
взята walk-forward из предыдущего окна, то есть строго из прошлого.

**Что допущено (не измерено).** Параметры филлов $A$ и $k$ из L2 не выводятся: нет tape и своих филлов.
Конкретно допущены: $A=5$, $k=1.5$ как базовая точка (и sweep по ним), $\gamma=0.01$,
$\sigma=1$ bp/$\sqrt{s}$ в модельных таблицах, funding $f=1$ bp, информированность $\pi=0.2$, прыжок
$J=2$ bp. Все эти числа — иллюстративные допущения, и каждый результат выше условен на них. Меняя $A$ и $k$
в один-два раза, мы меняем CE в разы (sweep: $29 \to 439$).

**Чувствительность.** К $k$ — сильная и монотонная (CE падает с ростом $k$). К $A$ — сильная и
монотонная (растёт с $A$). К $\sigma$ — skew растёт линейно по $\sigma$ (не $\sigma^2$: stationary-член
даёт корень из $\sigma^2$), а inventory variance от $\sigma$ не зависит вовсе. Порядок CE между политиками
хрупок: он зависит от выбранного $\sigma$ окна и от $(A,k)$.

**Пройденные тесты.** Симметрия филлов (0 против 5.6–67.3 при баге); risk-neutral оптимум $=1/k$;
inventory variance $=1/(2kc)$ независимо от $A,\sigma$; closed form GLFT против DP до 0.01 bp; prefix и
perturbation (нет look-ahead).

**Почему результат — верхняя граница.** Модель филлов — независимые Бернулли с пуассоновской
интенсивностью, постоянной $\sigma$ и мгновенным исполнением без latency и queue priority. В реальности
филлы кластеризуются (sweeps), $\sigma$ меняется в разы внутри суток, а лимит inventory маскирует adverse
selection. Все три эффекта играют против мейкера, поэтому полученные P&L и CE — **верхняя граница** того,
что можно ожидать вживую. Результат, не устойчивый к $A$ и $k$, — сам по себе вывод, а не артефакт.

**Что мерить дальше.** Чтобы получить $A$ и $k$ как оценки, а не допущения, нужны собственные ордера и
филлы с exposure times по дистанциям от mid: при $N=50$ филлов ошибка в $k$ — порядка 16%, при $N=1000$ —
около 3%. Из L2 этого не получить, поэтому следующий шаг — сбор собственного tape филлов.

## Выводы и ограничения

Модель маркет-мейкинга действительно превращает четыре числа ($\sigma$, $\gamma$, $A$, $k$) в две цены, и
её механика проста. Сложная часть — числа: из L2 мы можем измерить только $\sigma$ (и то непостоянную),
спред и глубину, а $A$ и $k$ — нет. Поэтому весь день проходил под знаком «результат условен на допущенные
$A,k$».

Главный содержательный результат: closed-form AS использует неверный горизонт $T-t$, из-за чего завышает
inventory skew примерно в 14 раз (при $\tau=60$ с), тогда как точное DP-решение и stationary GLFT дают
согласованный skew, не зависящий от горизонта. На 24/7-рынке горизонта нет, и правильная модель —
stationary. Симулятор на реальных mid-путях это подтверждает: GLFT и AS дают сопоставимый CE, а fixed —
широкий разброс при высоком сыром P&L.

Ограничения: $A,k$ — допущения; $\sigma$ предполагается постоянной (реально 4.6×
разброс); филлы — Бернулли без кластеризации, latency и queue priority; funding rate задан, а не измерен;
результат на фиксированном наборе окон (300 × 60 с). Всё это означает, что числа — верхняя граница, а
главная ценность дня не в уровнях P&L, а в том, какие из выводов выживают при смене $A$ и $k$, и в том, что
диагностировать adverse selection нужно по markout, а не по P&L на филл.